# 10-Model Evaluation on the Undirected Structural Branch

This notebook evaluates the next-report branch using the phase-0 Node2Vec caches from `node2vec-2.ipynb`.

Compared model families:

1. `BGE-M3 Only`
2. `Node2Vec (uncorrected)` undirected branch
3. `Node2Vec (corrected)` undirected branch
4. `BGE-M3 + Node2Vec (uncorrected)` undirected branch
5. `BGE-M3 + Node2Vec (corrected)` undirected branch
6. `GCN Only`
7. `GCN + BGE-M3 (rank fusion)`
8. `GraphSAGE Only`
9. `BGE pooled + Node2Vec (uncorrected)` undirected branch
10. `BGE pooled + Node2Vec (corrected)` undirected branch

Evaluation is based on the undirected structural split from phase 0. For ranking metrics, each undirected test edge `{u, v}` is expanded into two evaluation queries: `u -> v` and `v -> u`.

Important note: GCN, GraphSAGE, and GCN+BGE rank fusion should only be included if eval-3-specific GNN caches exist for this revised branch. Reusing older GNN caches trained on the previous split is disabled by default in this notebook.

Outputs:

- old metrics table (`Hit@1`, `Hit@10`, `Hit@50`, `Hit@100`, `MRR`, `NDCG`, `Recall`, `MAP`)
- new recommendation-quality table (`Popularity Lift`, `Gini`, `Aggregate Diversity`, `Novelty`, `ILD`)
- combined report-ready cache files under `cache/eval-3/`


In [ ]:
import os
import json
import math
import pickle
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

REPO_ROOT = r'C:\\programming\\github-repos\\graph-ending'
CACHE_DIR = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'eval-3')
PHASE0_CACHE = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'node2vec-2')
GCN_CACHE = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'gcn')
SAGE_CACHE = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'graphSAGE')
os.makedirs(CACHE_DIR, exist_ok=True)

DATA_PATH = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'data', 'data-embeddings.json')
BGE_PATH = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cap-embeddings', 'BAAI_bge-m3', 'master_embeddings.parquet')

TOP_K_RECS = 20
ALLOW_LEGACY_GNN_CACHE = False

print('Setup complete.')
print(f'Cache dir: {CACHE_DIR}')


## Step 1 - Load Data and Phase-0 Structural Branch


In [ ]:
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

nodes = data['nodes']
node_ids = [int(n['id']) for n in nodes]
node_id_set = set(node_ids)
id_to_title = {int(n['id']): n['title'] for n in nodes}
id_to_label = {int(n['id']): n.get('label', 'Unknown') for n in nodes}

G_full = nx.DiGraph()
for n in nodes:
    nid = int(n['id'])
    G_full.add_node(nid, title=n['title'], label=n.get('label', 'Unknown'))
for n in nodes:
    src = int(n['id'])
    for tgt in n.get('outlinks', []):
        tgt = int(tgt)
        if tgt in node_id_set:
            G_full.add_edge(src, tgt)
G_full_und = G_full.to_undirected()

with open(os.path.join(PHASE0_CACHE, 'phase0_metadata.json'), 'r', encoding='utf-8') as f:
    phase0_meta = json.load(f)
with open(os.path.join(PHASE0_CACHE, 'undirected_split.pkl'), 'rb') as f:
    split = pickle.load(f)
with open(os.path.join(PHASE0_CACHE, 'alpha_structural.pkl'), 'rb') as f:
    alpha_data = pickle.load(f)

E_train = split['E_train']
E_test = split['E_test']
eval_nodes = split['eval_nodes']
G_train_struct = split['G_train_struct']
alpha_structural = alpha_data['alpha']

# Expand undirected test edges into two evaluation directions.
eval_pairs = []
test_neighbors = defaultdict(set)
for u, v in E_test:
    u = int(u)
    v = int(v)
    eval_pairs.append((u, v))
    eval_pairs.append((v, u))
    test_neighbors[u].add(v)
    test_neighbors[v].add(u)

print(f'Full graph: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} directed edges')
print(f'Full undirected graph: {G_full_und.number_of_edges()} edges')
print(f'Undirected train edges: {len(E_train)}')
print(f'Undirected test edges : {len(E_test)}')
print(f'Expanded eval pairs   : {len(eval_pairs)}')
print(f'Eval nodes            : {len(eval_nodes)}')
print(f'Alpha structural      : {alpha_structural:.4f}')


## Step 2 - Load Embeddings and Build Model Matrices

The pooled-BGE variants use a simple mean-pooling compression from 1024 dimensions to 128 dimensions by averaging each contiguous block of 8 features. This gives a balanced 256-dimensional hybrid once concatenated with 128-dimensional Node2Vec.


In [ ]:
def l2_normalize(mat):
    return mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-8)

def mean_pool_bge(bge_matrix, target_dim=128):
    if bge_matrix.shape[1] % target_dim != 0:
        raise ValueError('BGE dimension must divide cleanly by target_dim for mean pooling.')
    block = bge_matrix.shape[1] // target_dim
    return bge_matrix.reshape(bge_matrix.shape[0], target_dim, block).mean(axis=2)

df_bge = pd.read_parquet(BGE_PATH).sort_values('id').reset_index(drop=True)
df_n2v_uncorr = pd.read_parquet(os.path.join(PHASE0_CACHE, 'node2vec_undirected_uncorrected.parquet')).sort_values('id').reset_index(drop=True)
df_n2v_corr = pd.read_parquet(os.path.join(PHASE0_CACHE, 'node2vec_undirected_corrected.parquet')).sort_values('id').reset_index(drop=True)

assert (df_bge['id'].astype(int).values == df_n2v_uncorr['id'].astype(int).values).all()
assert (df_bge['id'].astype(int).values == df_n2v_corr['id'].astype(int).values).all()

node_ids_sorted = df_bge['id'].astype(int).values
common_id_to_idx = {nid: i for i, nid in enumerate(node_ids_sorted)}

bge_raw = np.stack(df_bge['embedding'].values)
bge_emb = l2_normalize(bge_raw.copy())
bge_pooled = l2_normalize(mean_pool_bge(bge_raw, target_dim=128))

n2v_uncorr_raw = np.stack(df_n2v_uncorr['embedding'].values)
n2v_corr_raw = np.stack(df_n2v_corr['embedding'].values)
n2v_uncorr_emb = l2_normalize(n2v_uncorr_raw.copy())
n2v_corr_emb = l2_normalize(n2v_corr_raw.copy())

combined_uncorr = l2_normalize(np.concatenate([bge_raw, n2v_uncorr_raw], axis=1))
combined_corr = l2_normalize(np.concatenate([bge_raw, n2v_corr_raw], axis=1))
pooled_combined_uncorr = l2_normalize(np.concatenate([bge_pooled, n2v_uncorr_raw], axis=1))
pooled_combined_corr = l2_normalize(np.concatenate([bge_pooled, n2v_corr_raw], axis=1))

def load_optional_gnn_cache(preferred_path, legacy_path, allow_legacy=False):
    if os.path.exists(preferred_path):
        with open(preferred_path, 'rb') as f:
            return pickle.load(f), 'eval-3'
    if allow_legacy and os.path.exists(legacy_path):
        with open(legacy_path, 'rb') as f:
            return pickle.load(f), 'legacy'
    return None, 'missing'

gcn_data, gcn_cache_mode = load_optional_gnn_cache(
    os.path.join(CACHE_DIR, 'gcn_only_results_eval3.pkl'),
    os.path.join(GCN_CACHE, 'gcn_only_results.pkl'),
    allow_legacy=ALLOW_LEGACY_GNN_CACHE,
)
if gcn_data is not None:
    gcn_node_to_idx = {int(k): int(v) for k, v in gcn_data['node_to_idx'].items()}
    gcn_emb = l2_normalize(np.asarray(gcn_data['final_embeddings'], dtype=np.float64))
else:
    gcn_node_to_idx = None
    gcn_emb = None

sage_data, sage_cache_mode = load_optional_gnn_cache(
    os.path.join(CACHE_DIR, 'graphsage_results_eval3.pkl'),
    os.path.join(SAGE_CACHE, 'graphsage_results.pkl'),
    allow_legacy=ALLOW_LEGACY_GNN_CACHE,
)
if sage_data is not None:
    sage_node_to_idx = {int(k): int(v) for k, v in sage_data['node_to_idx'].items()}
    sage_emb = l2_normalize(np.asarray(sage_data['final_embeddings'], dtype=np.float64))
else:
    sage_node_to_idx = None
    sage_emb = None

print(f'BGE-M3: {bge_emb.shape}')
print(f'BGE pooled: {bge_pooled.shape}')
print(f'Node2Vec undirected uncorr: {n2v_uncorr_emb.shape}')
print(f'Node2Vec undirected corr  : {n2v_corr_emb.shape}')
if gcn_emb is not None:
    print(f'GCN embeddings            : {gcn_emb.shape} ({gcn_cache_mode})')
else:
    print('GCN embeddings            : unavailable for eval-3 branch')
if sage_emb is not None:
    print(f'GraphSAGE embeddings      : {sage_emb.shape} ({sage_cache_mode})')
else:
    print('GraphSAGE embeddings      : unavailable for eval-3 branch')


## Step 3 - Similarity Matrices


In [ ]:
def get_sim_matrix(emb, name):
    fpath = os.path.join(CACHE_DIR, f'sim_{name}.pkl')
    if os.path.exists(fpath):
        with open(fpath, 'rb') as f:
            return pickle.load(f)
    sim = cosine_similarity(emb)
    np.fill_diagonal(sim, 0.0)
    with open(fpath, 'wb') as f:
        pickle.dump(sim, f)
    return sim

sim_bge = get_sim_matrix(bge_emb, 'bge')
sim_n2v_uncorr = get_sim_matrix(n2v_uncorr_emb, 'n2v_undirected_uncorrected')
sim_n2v_corr = get_sim_matrix(n2v_corr_emb, 'n2v_undirected_corrected')
sim_comb_uncorr = get_sim_matrix(combined_uncorr, 'combined_undirected_uncorrected')
sim_comb_corr = get_sim_matrix(combined_corr, 'combined_undirected_corrected')
sim_gcn = get_sim_matrix(gcn_emb, 'gcn') if gcn_emb is not None else None
sim_sage = get_sim_matrix(sage_emb, 'sage') if sage_emb is not None else None
sim_pool_uncorr = get_sim_matrix(pooled_combined_uncorr, 'pooled_combined_uncorrected')
sim_pool_corr = get_sim_matrix(pooled_combined_corr, 'pooled_combined_corrected')

print('All similarity matrices ready.')


## Step 4 - Evaluation Helpers


In [ ]:
def evaluate_model(sim_matrix, id_to_idx, eval_pairs, k_values=[1, 10, 50, 100]):
    hits = {k: 0 for k in k_values}
    mrr = 0.0
    ndcg = 0.0
    recall_sum = 0.0
    ap_sum = 0.0
    n_evaluated = 0

    for src, tgt in eval_pairs:
        if src not in id_to_idx or tgt not in id_to_idx:
            continue
        src_idx = id_to_idx[src]
        tgt_idx = id_to_idx[tgt]
        sims = sim_matrix[src_idx]
        ranked_indices = np.argsort(-sims)
        rank = np.where(ranked_indices == tgt_idx)[0][0] + 1

        for k in k_values:
            if rank <= k:
                hits[k] += 1
        mrr += 1.0 / rank
        dcg = 1.0 / np.log2(rank + 1) if rank <= max(k_values) else 0.0
        idcg = 1.0 / np.log2(2)
        ndcg += dcg / idcg
        if rank <= max(k_values):
            recall_sum += 1.0
            ap_sum += 1.0 / rank
        n_evaluated += 1

    n = max(n_evaluated, 1)
    return {
        **{f'Hit@{k}': hits[k] / n for k in k_values},
        'MRR': mrr / n,
        'NDCG': ndcg / n,
        'Recall': recall_sum / n,
        'MAP': ap_sum / n,
        'n_eval_pairs': n_evaluated,
    }

def rank_fusion_evaluate(sim1, id1, sim2, id2, eval_pairs, k_values=[1, 10, 50, 100]):
    hits = {k: 0 for k in k_values}
    mrr = 0.0
    ndcg = 0.0
    recall_sum = 0.0
    ap_sum = 0.0
    n_evaluated = 0
    k_rrf = 60

    common_ids = sorted(set(id1.keys()) & set(id2.keys()))
    common_pos = {nid: i for i, nid in enumerate(common_ids)}
    idxs1 = np.array([id1[nid] for nid in common_ids], dtype=np.int32)
    idxs2 = np.array([id2[nid] for nid in common_ids], dtype=np.int32)

    for src, tgt in eval_pairs:
        if src not in id1 or src not in id2 or tgt not in common_pos:
            continue
        sims1 = sim1[id1[src]][idxs1]
        sims2 = sim2[id2[src]][idxs2]
        ranked1 = np.argsort(-sims1)
        ranked2 = np.argsort(-sims2)
        rankpos1 = np.empty(len(common_ids), dtype=np.int32)
        rankpos2 = np.empty(len(common_ids), dtype=np.int32)
        rankpos1[ranked1] = np.arange(1, len(common_ids) + 1)
        rankpos2[ranked2] = np.arange(1, len(common_ids) + 1)
        rrf_scores = 1.0 / (k_rrf + rankpos1) + 1.0 / (k_rrf + rankpos2)
        final_ranking = np.argsort(-rrf_scores)
        final_rankpos = np.empty(len(common_ids), dtype=np.int32)
        final_rankpos[final_ranking] = np.arange(1, len(common_ids) + 1)
        rank = final_rankpos[common_pos[tgt]]

        for k in k_values:
            if rank <= k:
                hits[k] += 1
        mrr += 1.0 / rank
        dcg = 1.0 / np.log2(rank + 1) if rank <= max(k_values) else 0.0
        idcg = 1.0 / np.log2(2)
        ndcg += dcg / idcg
        if rank <= max(k_values):
            recall_sum += 1.0
            ap_sum += 1.0 / rank
        n_evaluated += 1

    n = max(n_evaluated, 1)
    return {
        **{f'Hit@{k}': hits[k] / n for k in k_values},
        'MRR': mrr / n,
        'NDCG': ndcg / n,
        'Recall': recall_sum / n,
        'MAP': ap_sum / n,
        'n_eval_pairs': n_evaluated,
    }

def get_topk_lists(sim_matrix, id_to_idx, query_nodes, top_k=20):
    lists = {}
    for src in query_nodes:
        if src not in id_to_idx:
            continue
        src_idx = id_to_idx[src]
        ranked = np.argsort(-sim_matrix[src_idx])
        recs = []
        for idx in ranked:
            if idx == src_idx:
                continue
            recs.append(idx)
            if len(recs) == top_k:
                break
        lists[src] = recs
    return lists

def get_topk_lists_rrf(sim1, id1, sim2, id2, query_nodes, top_k=20):
    common_ids = sorted(set(id1.keys()) & set(id2.keys()))
    common_pos = {nid: i for i, nid in enumerate(common_ids)}
    idxs1 = np.array([id1[nid] for nid in common_ids], dtype=np.int32)
    idxs2 = np.array([id2[nid] for nid in common_ids], dtype=np.int32)
    lists = {}
    k_rrf = 60
    for src in query_nodes:
        if src not in id1 or src not in id2:
            continue
        sims1 = sim1[id1[src]][idxs1]
        sims2 = sim2[id2[src]][idxs2]
        ranked1 = np.argsort(-sims1)
        ranked2 = np.argsort(-sims2)
        rankpos1 = np.empty(len(common_ids), dtype=np.int32)
        rankpos2 = np.empty(len(common_ids), dtype=np.int32)
        rankpos1[ranked1] = np.arange(1, len(common_ids) + 1)
        rankpos2[ranked2] = np.arange(1, len(common_ids) + 1)
        rrf_scores = 1.0 / (k_rrf + rankpos1) + 1.0 / (k_rrf + rankpos2)
        final_ranking = np.argsort(-rrf_scores)
        recs = []
        for pos in final_ranking:
            nid = common_ids[pos]
            if nid == src:
                continue
            recs.append(nid)
            if len(recs) == top_k:
                break
        lists[src] = recs
    return lists

def recommendation_metrics(topk_lists, idx_to_node, semantic_matrix, semantic_id_to_idx, graph_und, top_k=20):
    all_recs = []
    ild_values = []
    popularity_scores = []
    novelty_scores = []
    valid_lists = 0

    candidate_nodes = [nid for nid in semantic_id_to_idx.keys() if nid in graph_und]
    candidate_degrees = np.array([max(graph_und.degree(nid), 0) for nid in candidate_nodes], dtype=np.float64)
    baseline_popularity = candidate_degrees.mean() if len(candidate_degrees) else 1.0
    total_degree_mass = candidate_degrees.sum() + len(candidate_degrees)

    for src, rec_indices in topk_lists.items():
        rec_nodes = []
        for idx in rec_indices:
            if isinstance(idx, (np.integer, int)) and idx in idx_to_node:
                rec_nodes.append(int(idx_to_node[idx]))
            elif isinstance(idx, (np.integer, int)) and idx in semantic_id_to_idx:
                rec_nodes.append(int(idx))
        if not rec_nodes:
            continue
        valid_lists += 1
        all_recs.extend(rec_nodes)

        degs = np.array([max(graph_und.degree(nid), 0) for nid in rec_nodes], dtype=np.float64)
        popularity_scores.append(degs.mean())
        novelty_scores.append(np.mean([-np.log2((deg + 1.0) / total_degree_mass) for deg in degs]))

        sem_indices = [semantic_id_to_idx[nid] for nid in rec_nodes if nid in semantic_id_to_idx]
        if len(sem_indices) >= 2:
            sub_sim = semantic_matrix[np.ix_(sem_indices, sem_indices)]
            dists = 1.0 - sub_sim
            tri = dists[np.triu_indices_from(dists, k=1)]
            if len(tri):
                ild_values.append(float(tri.mean()))

    aggregate_diversity = len(set(all_recs))
    popularity_lift = (np.mean(popularity_scores) / baseline_popularity) if popularity_scores else 0.0
    novelty = float(np.mean(novelty_scores)) if novelty_scores else 0.0
    ild = float(np.mean(ild_values)) if ild_values else 0.0

    if all_recs:
        counts = np.array(sorted(Counter(all_recs).values()), dtype=np.float64)
        n = len(counts)
        gini = float((np.sum((2 * np.arange(1, n + 1) - n - 1) * counts)) / (n * counts.sum())) if counts.sum() > 0 else 0.0
    else:
        gini = 0.0

    return {
        'Popularity Lift': float(popularity_lift),
        'Gini Index': float(gini),
        'Aggregate Diversity': int(aggregate_diversity),
        'Novelty': float(novelty),
        'Intra-List Distance': float(ild),
        'n_recommendation_lists': int(valid_lists),
        'top_k': int(top_k),
    }

print('Evaluation helpers ready.')


## Step 5 - Evaluate All 10 Models


In [ ]:
models = [
    {'name': '1. BGE-M3 Only', 'sim': sim_bge, 'id_to_idx': common_id_to_idx, 'kind': 'direct', 'idx_to_node': {i: nid for nid, i in common_id_to_idx.items()}},
    {'name': '2. Node2Vec (uncorrected)', 'sim': sim_n2v_uncorr, 'id_to_idx': common_id_to_idx, 'kind': 'direct', 'idx_to_node': {i: nid for nid, i in common_id_to_idx.items()}},
    {'name': '3. Node2Vec (corrected)', 'sim': sim_n2v_corr, 'id_to_idx': common_id_to_idx, 'kind': 'direct', 'idx_to_node': {i: nid for nid, i in common_id_to_idx.items()}},
    {'name': '4. BGE-M3 + Node2Vec (uncorrected)', 'sim': sim_comb_uncorr, 'id_to_idx': common_id_to_idx, 'kind': 'direct', 'idx_to_node': {i: nid for nid, i in common_id_to_idx.items()}},
    {'name': '5. BGE-M3 + Node2Vec (corrected)', 'sim': sim_comb_corr, 'id_to_idx': common_id_to_idx, 'kind': 'direct', 'idx_to_node': {i: nid for nid, i in common_id_to_idx.items()}},
    {'name': '9. BGE pooled + Node2Vec (uncorrected)', 'sim': sim_pool_uncorr, 'id_to_idx': common_id_to_idx, 'kind': 'direct', 'idx_to_node': {i: nid for nid, i in common_id_to_idx.items()}},
    {'name': '10. BGE pooled + Node2Vec (corrected)', 'sim': sim_pool_corr, 'id_to_idx': common_id_to_idx, 'kind': 'direct', 'idx_to_node': {i: nid for nid, i in common_id_to_idx.items()}},
]

if gcn_emb is not None:
    models.insert(5, {'name': '6. GCN Only', 'sim': sim_gcn, 'id_to_idx': gcn_node_to_idx, 'kind': 'direct', 'idx_to_node': {v: k for k, v in gcn_node_to_idx.items()}})
    models.insert(6, {'name': '7. GCN + BGE-M3 (rank fusion)', 'sim1': sim_gcn, 'id1': gcn_node_to_idx, 'sim2': sim_bge, 'id2': common_id_to_idx, 'kind': 'rrf', 'idx_to_node': None})
if sage_emb is not None:
    insert_at = 7 if gcn_emb is not None else 5
    models.insert(insert_at, {'name': '8. GraphSAGE Only', 'sim': sim_sage, 'id_to_idx': sage_node_to_idx, 'kind': 'direct', 'idx_to_node': {v: k for k, v in sage_node_to_idx.items()}})

old_metric_rows = []
new_metric_rows = []
full_results = {}

print('Evaluating all 10 models...')
print('=' * 72)

for model in models:
    name = model['name']
    print(name)
    if model['kind'] == 'direct':
        old_metrics = evaluate_model(model['sim'], model['id_to_idx'], eval_pairs)
        topk_lists = get_topk_lists(model['sim'], model['id_to_idx'], sorted(eval_nodes), top_k=TOP_K_RECS)
    else:
        old_metrics = rank_fusion_evaluate(model['sim1'], model['id1'], model['sim2'], model['id2'], eval_pairs)
        topk_lists = get_topk_lists_rrf(model['sim1'], model['id1'], model['sim2'], model['id2'], sorted(eval_nodes), top_k=TOP_K_RECS)
    new_metrics = recommendation_metrics(topk_lists, model.get('idx_to_node') or {}, sim_bge, common_id_to_idx, G_full_und, top_k=TOP_K_RECS)

    old_metric_rows.append({'Model': name, **old_metrics})
    new_metric_rows.append({'Model': name, **new_metrics})
    full_results[name] = {'old_metrics': old_metrics, 'new_metrics': new_metrics}

    print(f"  Hit@10: {old_metrics['Hit@10']:.4f}, MRR: {old_metrics['MRR']:.4f}, Aggregate Diversity: {new_metrics['Aggregate Diversity']}")

print('=' * 72)
print('Evaluation complete.')


## Step 6 - Save Tables and Artifacts


In [ ]:
df_old = pd.DataFrame(old_metric_rows).sort_values('Hit@10', ascending=False).reset_index(drop=True)
df_new = pd.DataFrame(new_metric_rows).sort_values('Aggregate Diversity', ascending=False).reset_index(drop=True)
df_combined = df_old.merge(df_new, on='Model', suffixes=('_old', '_new'))

df_old.to_csv(os.path.join(CACHE_DIR, 'old_metrics.csv'), index=False)
df_new.to_csv(os.path.join(CACHE_DIR, 'new_metrics.csv'), index=False)
df_combined.to_csv(os.path.join(CACHE_DIR, 'combined_metrics.csv'), index=False)

with open(os.path.join(CACHE_DIR, 'full_results.pkl'), 'wb') as f:
    pickle.dump(full_results, f)
with open(os.path.join(CACHE_DIR, 'full_results.json'), 'w', encoding='utf-8') as f:
    json.dump(full_results, f, indent=2)

run_meta = {
    'phase0_branch': phase0_meta['branch_name'],
    'n_models': len(models),
    'top_k_recommendation_metrics': TOP_K_RECS,
    'n_eval_pairs': len(eval_pairs),
    'n_eval_nodes': len(eval_nodes),
    'n_test_edges_undirected': len(E_test),
    'alpha_structural': float(alpha_structural),
    'pooling_method': 'mean pooling over contiguous 8-feature BGE blocks (1024 -> 128)',
    'gcn_cache_mode': gcn_cache_mode,
    'graphsage_cache_mode': sage_cache_mode,
    'allow_legacy_gnn_cache': ALLOW_LEGACY_GNN_CACHE,
    'models_evaluated': [m['name'] for m in models],
    'artifacts': {
        'old_metrics_csv': os.path.join(CACHE_DIR, 'old_metrics.csv'),
        'new_metrics_csv': os.path.join(CACHE_DIR, 'new_metrics.csv'),
        'combined_metrics_csv': os.path.join(CACHE_DIR, 'combined_metrics.csv'),
        'full_results_pkl': os.path.join(CACHE_DIR, 'full_results.pkl'),
        'full_results_json': os.path.join(CACHE_DIR, 'full_results.json'),
    },
}
with open(os.path.join(CACHE_DIR, 'run_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(run_meta, f, indent=2)

print('Saved all eval-3 artifacts.')


## Step 7 - Preview Tables


In [ ]:
print('Old metrics (sorted by Hit@10):')
display(df_old)

print('New recommendation metrics (sorted by Aggregate Diversity):')
display(df_new)
